In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# =========================
# PATH CONFIGURATION
# =========================

# COLAB PATH
data_path = "/content/data-portfolio/projects/north_texas_housing/data/raw/zillow/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

# LOCAL / GIT PATH
# data_path = "data/raw/zillow/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

# =========================
# LOAD DATASET
# =========================
df = pd.read_csv(data_path)

# =========================
# INITIAL CHECKS
# =========================
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())

# =========================
# TRANSFORM TO LONG FORMAT
# =========================
id_vars = [
    "RegionID",
    "SizeRank",
    "RegionName",
    "RegionType",
    "StateName",
    "State",
    "City",
    "Metro",
    "CountyName"
]

date_columns = [col for col in df.columns if col not in id_vars]

df_long = df.melt(
    id_vars=id_vars,
    value_vars=date_columns,
    var_name="date",
    value_name="home_value"
)

# Convert date column
df_long["date"] = pd.to_datetime(df_long["date"])

# Remove nulls in home_value
df_long = df_long.dropna(subset=["home_value"])

# =========================
# FILTER TEXAS
# =========================
df_tx = df_long[df_long["State"] == "TX"].copy()

# =========================
# FILTER TARGET CITIES
# =========================
target_cities = ["McKinney", "Frisco", "Allen", "Melissa"]
df_ntx = df_tx[df_tx["City"].isin(target_cities)].copy()

print(df_ntx["City"].value_counts())

# =========================
# CREATE AVERAGE TREND BY CITY
# =========================
city_trend = (
    df_ntx.groupby(["date", "City"], as_index=False)["home_value"]
    .mean()
)

# Sort before calculating YoY
city_trend = city_trend.sort_values(["City", "date"])

# =========================
# CALCULATE YEAR-OVER-YEAR CHANGE
# =========================
city_trend["yoy_change_pct"] = (
    city_trend.groupby("City")["home_value"]
    .pct_change(periods=12) * 100
)

print(city_trend.head(20))

# =========================
# PLOT 1 - AVERAGE HOME VALUE TREND
# =========================
plt.figure(figsize=(12, 6))
sns.lineplot(data=city_trend, x="date", y="home_value", hue="City")
plt.title("Average Home Value Trend - North Texas Cities")
plt.xlabel("Date")
plt.ylabel("Home Value")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# =========================
# PLOT 2 - YOY GAIN / LOSS (%)
# =========================
plt.figure(figsize=(12, 6))
sns.lineplot(data=city_trend, x="date", y="yoy_change_pct", hue="City")
plt.axhline(0, linestyle="--")
plt.title("Year-over-Year Home Value Change (%) - North Texas Cities")
plt.xlabel("Date")
plt.ylabel("YoY Change (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# =========================
# OPTIONAL - LATEST YOY BY CITY
# =========================
latest_yoy = (
    city_trend.dropna(subset=["yoy_change_pct"])
    .groupby("City", as_index=False)
    .tail(1)
    .sort_values("yoy_change_pct", ascending=False)
)

print(latest_yoy[["City", "date", "home_value", "yoy_change_pct"]])

FileNotFoundError: [Errno 2] No such file or directory: '/content/data-portfolio/projects/north_texas_housing/data/raw/zillow/Zip_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv'